In [4]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.stats.multitest

In [6]:
count_matrix = pd.read_csv(
    "/Users/michael/Git/Penelope_Dave_Maize_meristem/Post_CSHL_requests/Data/vst.norm.counts.combatseq.Feb2026_less_stringent_no_fea3.csv",
    index_col=0,
)
metadata = pd.read_csv(
    "/Users/michael/Git/Penelope_Dave_Maize_meristem/Post_CSHL_requests/Data/NAM_RNAseq_metadata_IM_outliers_removed_no_fea3.csv",
    index_col=0,
)

In [5]:
metadata.head(40)

,condition,size,size_bin,Field,type,KRN,Total_kernel_number,IM_height,Cob_length,Cob_diameter,Number_kernels_row,twenty_kernel_weight,SAM_radius,SAM_height
sampName,,,,,,,,,,,,,,
B73_1b,B73,454.888985,c,WN21,stiff,17.27,186.41,243.515901,139.88,29.79,24.78,4.56,82.9954,129.5369
B73_2b,B73,454.888985,c,WN21,stiff,17.27,186.41,243.515901,139.88,29.79,24.78,4.56,82.9954,129.5369
B73_1,B73,454.888985,c,WN20,stiff,17.27,186.41,243.515901,139.88,29.79,24.78,4.56,82.9954,129.5369
B73_2,B73,454.888985,c,WN20,stiff,17.27,186.41,243.515901,139.88,29.79,24.78,4.56,82.9954,129.5369
B73_3,B73,454.888985,c,WN20,stiff,17.27,186.41,243.515901,139.88,29.79,24.78,4.56,82.9954,129.5369
B104_1,B104,351.818059,b,WN20,stiff,14.00,NaN,237.214056,NaN,NaN,NaN,NaN,NaN,NaN
B104_2,B104,351.818059,b,WN20,stiff,14.00,NaN,237.214056,NaN,NaN,NaN,NaN,NaN,NaN
B104_3,B104,351.818059,b,WN20,stiff,14.00,NaN,237.214056,NaN,NaN,NaN,NaN,NaN,NaN
CML103_2b,CML103,371.815220,b,WN21,tropical,13.50,204.88,245.164838,153.58,26.96,23.11,5.07,70.7104,116.6321


In [7]:
# Get unique NAM lines and their trait values (size and KRN)
# Each NAM line has the same size/KRN across replicates, so take the first value
trait_per_line = metadata.groupby("condition")[["size", "KRN"]].first()
nam_lines = trait_per_line.index.tolist()

# Average gene expression across replicates for each NAM line
mean_expr_per_line = pd.DataFrame(index=count_matrix.columns)
for line in nam_lines:
    samples = metadata.index[metadata["condition"] == line]
    mean_expr_per_line[line] = count_matrix.loc[count_matrix.index.isin(samples)].mean(
        axis=0
    )

print(f"NAM lines: {len(nam_lines)}, Genes: {mean_expr_per_line.shape[0]}")
mean_expr_per_line

NAM lines: 24, Genes: 11421


,B104,B73,CML103,CML228,CML277,CML322,CML333,CML52,CML69,Hp301,...,Mo17,Mo18W,Ms71,NC350,NC358,Oh43,Oh7B,P39,Tx303,Tzi8
GRMZM2G059865,4.643796,6.978318,6.267381,6.281153,7.081814,6.517371,7.955359,7.776990,6.167865,6.752463,...,4.827783,6.054250,7.373471,6.750694,6.928369,6.692988,6.074435,6.912360,6.783960,7.388492
GRMZM2G330436,4.260174,4.250622,4.204215,4.633932,4.238562,5.307441,4.658502,4.596785,4.402143,4.448023,...,4.376558,4.374520,4.243954,4.122010,4.247574,4.442333,4.343500,3.761094,3.827122,4.279082
GRMZM2G032104,3.495499,4.039613,3.994942,4.097944,3.742706,4.369744,3.541834,3.732605,3.490153,4.162394,...,3.062123,3.968035,4.233269,4.361454,4.029220,4.172025,4.235439,4.215912,4.521392,3.643934
GRMZM2G073979,4.724129,4.492320,4.559392,4.473079,4.755866,4.756244,4.492178,4.697698,4.887898,4.362162,...,5.143630,4.810376,4.726480,4.170362,4.074746,4.399220,4.480340,3.983537,4.149706,4.444917
GRMZM2G374779,4.510866,3.744004,3.518976,3.584509,3.950773,3.713347,2.831533,3.561075,3.971326,3.757999,...,4.803676,4.212787,3.615082,3.431136,3.696439,3.945538,4.252398,3.386869,4.078514,3.002881
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ATPI,2.731885,2.502111,2.706570,3.130503,2.819604,2.999553,3.069430,2.902688,2.685085,2.938616,...,2.934608,2.509323,2.438207,3.434395,2.749448,2.839470,2.736534,2.856087,3.058450,2.632322
RPS11.3,3.425100,3.321286,3.483623,3.342520,3.714201,3.046820,3.518749,3.381000,3.412796,3.142285,...,3.992532,3.223170,3.734538,3.356126,2.557611,3.644840,3.121923,3.255465,3.959486,3.000971
RPL14,2.995532,3.391854,2.576275,2.880169,3.175390,2.885775,3.368505,2.422159,3.109989,2.963745,...,2.808350,3.060652,3.267715,3.471803,2.672411,3.281878,2.711257,3.009709,2.125010,2.665142
RPS3.1,3.134943,3.061388,3.121782,2.899258,3.047567,2.859913,3.278998,2.869133,3.380065,2.799059,...,3.570203,3.010024,3.125252,2.854846,2.945576,3.308776,2.635711,3.145499,3.318425,2.765927


In [8]:
# Calculate Pearson correlation + p-value for each gene vs Size and vs KRN
size_values = trait_per_line.loc[nam_lines, "size"]
krn_values = trait_per_line.loc[nam_lines, "KRN"]

results = []
for gene in mean_expr_per_line.index:
    expr = mean_expr_per_line.loc[gene, nam_lines]

    size_corr, size_p = stats.pearsonr(expr, size_values)
    krn_corr, krn_p = stats.pearsonr(expr, krn_values)

    results.append(
        {
            "Gene": gene,
            "Pearson_Corr_Size": size_corr,
            "P_Value_Size": size_p,
            "Pearson_Corr_KRN": krn_corr,
            "P_Value_KRN": krn_p,
        }
    )

results_df = pd.DataFrame(results).set_index("Gene")
results_df = results_df.dropna()
print(f"Genes with valid correlations: {len(results_df)}")
results_df

Genes with valid correlations: 11421


,Pearson_Corr_Size,P_Value_Size,Pearson_Corr_KRN,P_Value_KRN
Gene,,,,
GRMZM2G059865,-0.194894,0.361450,0.024681,0.908861
GRMZM2G330436,-0.269736,0.202424,-0.007858,0.970929
GRMZM2G032104,0.234587,0.269865,0.528170,0.007979
GRMZM2G073979,0.085222,0.692155,-0.240537,0.257551
GRMZM2G374779,-0.052914,0.806024,-0.082492,0.701565
...,...,...,...,...
ATPI,-0.007620,0.971811,-0.285697,0.175956
RPS11.3,0.001690,0.993747,-0.205442,0.335520
RPL14,0.339812,0.104242,0.222364,0.296324


In [9]:
# Benjamini-Hochberg FDR correction for both sets of p-values
_, results_df["BH_Corrected_P_Value_Size"] = statsmodels.stats.multitest.fdrcorrection(
    results_df["P_Value_Size"]
)
_, results_df["BH_Corrected_P_Value_KRN"] = statsmodels.stats.multitest.fdrcorrection(
    results_df["P_Value_KRN"]
)

# Reorder columns for clarity
results_df = results_df[
    [
        "Pearson_Corr_Size",
        "P_Value_Size",
        "BH_Corrected_P_Value_Size",
        "Pearson_Corr_KRN",
        "P_Value_KRN",
        "BH_Corrected_P_Value_KRN",
    ]
]

print(
    f"Significant genes (Size, BH < 0.05): {(results_df['BH_Corrected_P_Value_Size'] < 0.05).sum()}"
)
print(
    f"Significant genes (KRN, BH < 0.05): {(results_df['BH_Corrected_P_Value_KRN'] < 0.05).sum()}"
)
results_df.sort_values("P_Value_Size")

Significant genes (Size, BH < 0.05): 0
Significant genes (KRN, BH < 0.05): 0


,Pearson_Corr_Size,P_Value_Size,BH_Corrected_P_Value_Size,Pearson_Corr_KRN,P_Value_KRN,BH_Corrected_P_Value_KRN
Gene,,,,,,
GRMZM2G153766,0.759450,0.000017,0.191847,0.376044,0.070137,0.998707
GRMZM2G036905,0.729798,0.000052,0.254574,0.174012,0.416102,0.998707
GRMZM2G123585,-0.722503,0.000067,0.254574,-0.456872,0.024809,0.998707
GRMZM2G105787,0.668204,0.000359,0.998427,0.182404,0.393614,0.998707
GRMZM2G038898,0.634925,0.000859,0.998427,0.485891,0.016075,0.998707
...,...,...,...,...,...,...
GRMZM2G048482,0.000170,0.999372,0.999638,0.153997,0.472476,0.998707
GRMZM2G170842,-0.000169,0.999375,0.999638,0.286547,0.174617,0.998707
GRMZM2G108355,-0.000085,0.999686,0.999797,0.079774,0.710979,0.998707


In [10]:
results_df.to_csv(
    "/Users/michael/Git/Penelope_Dave_Maize_meristem/Post_CSHL_requests/Data/penelope_requested_analysis_krn_and_size.csv",
    index=True,
)